# 2 · Descobrir quais dados existem

**O que você vai aprender:** a consultar o catálogo do DATASUS antes de baixar
qualquer coisa — que estados, anos e meses estão disponíveis em cada sistema.
**Tempo estimado:** 3 minutos.

Este é o passo que evita o erro mais comum: pedir um período que não existe e
receber uma tabela vazia, sem nenhuma mensagem de erro.

## Preparação

Toda análise começa por aqui: instalar as bibliotecas e aplicar o `nest_asyncio`,
sem o qual a PySUS não funciona dentro de um notebook.

In [2]:
%pip install pysus nest_asyncio -q
import nest_asyncio
nest_asyncio.apply()
print("Ambiente pronto.")

Note: you may need to restart the kernel to use updated packages.
Ambiente pronto.


## Os sistemas disponíveis

| Sistema | O que contém | Recorte |
|---|---|---|
| `sinan` | Doenças de notificação (dengue, tuberculose, violência…) | Nacional, por ano |
| `sim` | Óbitos (mortalidade) | Estado e ano |
| `sinasc` | Nascidos vivos | Estado e ano |
| `sih` | Internações hospitalares | Estado, ano e mês |
| `sia` | Produção ambulatorial | Estado, ano e mês |
| `cnes` | Estabelecimentos, leitos, profissionais | Estado, ano e mês |
| `pni` | Vacinação | Estado e ano |
| `ibge` | População (para calcular taxas) | Ano |

## Consultando o catálogo

A função `list_files` responde em poucos segundos e **não baixa dados** — ela só
lista o que existe. Vamos ver os anos disponíveis do SIM (óbitos) no Paraná:

In [3]:
from pysus import list_files

catalogo = list_files(dataset="sim", state="PR")

anos = sorted({int(a) for a in catalogo["year"].dropna()})
print(f"Arquivos: {len(catalogo)}")
print(f"Anos disponíveis para o Paraná: {anos}")

Arquivos: 26
Anos disponíveis para o Paraná: [1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2011, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


## Bases mensais: cuidado com as lacunas

Nas bases mensais (SIH, SIA, CNES), nem todo mês está publicado. Veja as
internações (SIH) do Paraná em 2024:

In [4]:
catalogo_sih = list_files(dataset="sih", state="PR", year=2024)

# O nome do arquivo traz o grupo (2 letras), a UF e o período (AAMM)
arquivos = sorted(nome.split("\\")[-1] for nome in catalogo_sih["name"])
print(f"Arquivos encontrados: {len(arquivos)}")
for nome in arquivos[:14]:
    print("  ", nome)

Arquivos encontrados: 12
   RDPR2408.parquet
   RJPR2412.parquet
   SPPR2401.parquet
   SPPR2402.parquet
   SPPR2403.parquet
   SPPR2404.parquet
   SPPR2405.parquet
   SPPR2406.parquet
   SPPR2407.parquet
   SPPR2409.parquet
   SPPR2410.parquet
   SPPR2411.parquet


Repare que cada arquivo começa com duas letras — o **grupo**:

- `RD` = AIH reduzida (a mais usada: uma linha por internação)
- `SP` = serviços profissionais
- `RJ` = AIH rejeitada
- `ER` = erros

E os quatro números finais são ano e mês (`2403` = março de 2024). Se um mês
não aparecer nessa lista, ele ainda não foi publicado.

## Uma função para conferir antes de baixar

Vale guardar esta função: ela avisa se o período existe **antes** de você
gastar tempo com um download que voltaria vazio.

In [5]:
def existe_no_catalogo(dataset, **filtros):
    """Diz se há arquivos para o recorte pedido."""
    achados = list_files(dataset=dataset, **filtros)
    if len(achados) == 0:
        print(f"❌ Nada encontrado em {dataset} com {filtros}")
        return False
    print(f"✅ {len(achados)} arquivo(s) em {dataset} com {filtros}")
    return True


existe_no_catalogo("cnes", state="PR", year=2024, month=12)
existe_no_catalogo("sim", state="PR", year=2030)

✅ 12 arquivo(s) em cnes com {'state': 'PR', 'year': 2024, 'month': 12}


❌ Nada encontrado em sim com {'state': 'PR', 'year': 2030}


False

## Quais doenças o SINAN tem

O SINAN é organizado por agravo, com um código de quatro letras. Veja o que
existe para 2024:

In [6]:
catalogo_sinan = list_files(dataset="sinan", year=2024)

agravos = sorted({nome.split("\\")[-1][:4] for nome in catalogo_sinan["name"]})
print(f"{len(agravos)} agravos com dados em 2024:\n")
print(", ".join(agravos))

56 agravos com dados em 2024:

ACBI, ACGR, AIDA, AIDC, ANIM, ANTR, BOTU, CANC, CHAG, CHIK, COLE, COQU, DCRJ, DENG, DERM, DIFT, ESQU, EXAN, FMAC, FTIF, HANS, HANT, HIVA, HIVC, HIVE, HIVG, IEXO, LEIV, LEPT, LERD, LTAN, MALA, MENI, MENT, MPX_, NTRA, PAIR, PEST, PFAN, PNEU, RAIV, ROTA, SDTA, SIFA, SIFC, SIFG, SRCB, TETA, TETN, TOXC, TOXG, TRAC, TUBE, VARC, VIOL, ZIKA


Os mais procurados: `DENG` (dengue), `CHIK` (chikungunya), `ZIKA`, `TUBE`
(tuberculose), `HANS` (hanseníase), `LEPT` (leptospirose), `MALA` (malária),
`VIOL` (violência interpessoal/autoprovocada), `ANIM` (acidentes por animais
peçonhentos).

## Próximo passo

Agora que você sabe consultar o catálogo, siga para os exemplos completos nas
pastas `CNES/`, `SINAN/`, `SIM/` e `SINASC/`.

---
*Notebook do projeto [PySusNoCode](https://github.com/cartaproale/PySusNoCode) —
um produto [Kraemer Academy](https://kraemeracademy.net).
Validado com dados reais do DATASUS.*